# 

# Desafio 1: a classe Livro

In [2]:
class Livro:
    def __init__(self, isbn, titulo, autor, ano, exemplares):
        self.isbn = isbn
        self.titulo = titulo
        self.autor = autor
        self.ano = ano
        self._exemplares = 0
        self.exemplares = exemplares  # passa pelo setter (validação)

    @property
    def exemplares(self):
        return self._exemplares

    @exemplares.setter
    def exemplares(self, valor):
        if valor >= 0:
            self._exemplares = valor
        else:
            print("Número de exemplares inválido")

    def emprestar(self):
        if self._exemplares > 0:
            self._exemplares -= 1
            return True
        return False

    def devolver(self):
        self._exemplares += 1

    def __str__(self):
        return f"{self.titulo} — {self.autor} ({self.ano}) · {self._exemplares} ex."

    def __repr__(self):
        return f"Livro('{self.isbn}')"

    def __eq__(self, outro):
        if not isinstance(outro, Livro):
            return NotImplemented
        return self.isbn == outro.isbn

    def __hash__(self):
        return hash(self.isbn)


print("--- Desafio 1 ---")
l1 = Livro("978-972-0-04001-1", "Os Maias", "Eça de Queirós", 1888, 1)
l2 = Livro("978-972-0-04001-1", "Os Maias (2.ª ed.)", "Eça de Queirós", 1888, 5)
l1.exemplares = -2
print(l1)
print(l1.emprestar(), l1.emprestar())
l1.devolver()
print(l1)
print(l1 == l2, [l1, l2])

--- Desafio 1 ---
Número de exemplares inválido
Os Maias — Eça de Queirós (1888) · 1 ex.
True False
Os Maias — Eça de Queirós (1888) · 1 ex.
True [Livro('978-972-0-04001-1'), Livro('978-972-0-04001-1')]


# Desafio 2: a classe Biblioteca

In [4]:
class Biblioteca:
    def __init__(self):
        self._livros = []

    def adicionar(self, livro):
        if livro in self._livros:
            print(f"Já existe: {livro.isbn}")
            return False
        self._livros.append(livro)
        return True

    def disponiveis(self):
        for livro in self._livros:
            if livro.exemplares > 0:
                yield livro

    def todos(self):
        for livro in self._livros:
            yield livro

    def quantos(self):
        return len(self._livros)

    def total_exemplares(self):
        return sum(livro.exemplares for livro in self._livros)

    def procurar_autor(self, texto):
        return [l.titulo for l in self._livros if texto.casefold() in l.autor.casefold()]


b = Biblioteca()
b.adicionar(Livro("978-972-0-04002-8", "Memorial do Convento", "José Saramago", 1982, 2))
b.adicionar(Livro("978-972-0-04003-5", "Ensaio sobre a Cegueira", "José Saramago", 1995, 0))
b.adicionar(Livro("978-972-0-04002-8", "Memorial do Convento", "José Saramago", 1982, 9))
for livro in b.disponiveis():
    print(livro)
print(b.total_exemplares())
print(b.procurar_autor("saramago"))

Já existe: 978-972-0-04002-8
Memorial do Convento — José Saramago (1982) · 2 ex.
2
['Memorial do Convento', 'Ensaio sobre a Cegueira']


# Desafio 3: carregar os livros de um ficheiro CSV

In [5]:
import csv

dados = """978-972-0-04001-1,Os Maias,Eça de Queirós,1888,3
978-972-0-04002-8,Memorial do Convento,José Saramago,1982,2
978-972-0-04003-5,Ensaio sobre a Cegueira,José Saramago,1995,0
978-972-0-04004-2,A Cidade e as Serras,Eça de Queirós,1901,1
978-972-0-04005-9,Mensagem,Fernando Pessoa,1934,dois
978-972-0-04006-6,O Ano da Morte de Ricardo Reis,José Saramago,1984,4
978-972-0-04007-3,Livro do Desassossego,Fernando Pessoa,sem data,2
"""

with open("livros.csv", "w", encoding="utf-8", newline="") as f:
    f.write(dados)


def carregar_csv(caminho):
    bib = Biblioteca()
    ignoradas = 0
    campos = ["isbn", "titulo", "autor", "ano", "exemplares"]
    with open(caminho, encoding="utf-8", newline="") as f:
        for linha in csv.DictReader(f, fieldnames=campos):
            try:
                livro = Livro(linha["isbn"], linha["titulo"], linha["autor"],
                              int(linha["ano"]), int(linha["exemplares"]))
            except ValueError:
                ignoradas += 1
                continue
            bib.adicionar(livro)
    return bib, ignoradas


bib, ignoradas = carregar_csv("livros.csv")
print(bib.quantos(), "livros carregados;", ignoradas, "linhas ignoradas")
print(bib.total_exemplares())

5 livros carregados; 2 linhas ignoradas
10


# Desafio 4: guardar a biblioteca numa base de dados SQLite

In [7]:
import sqlite3


def guardar_sqlite(bib, caminho):
    con = sqlite3.connect(caminho)
    try:
        con.execute("""CREATE TABLE IF NOT EXISTS livros (
                           isbn TEXT PRIMARY KEY,
                           titulo TEXT,
                           autor TEXT,
                           ano INTEGER,
                           exemplares INTEGER)""")
        con.execute("DELETE FROM livros")
        con.executemany(
            "INSERT INTO livros VALUES (?, ?, ?, ?, ?)",
            [(l.isbn, l.titulo, l.autor, l.ano, l.exemplares) for l in bib.todos()])
        con.commit()
    finally:
        con.close()


guardar_sqlite(bib, "biblioteca.db")
guardar_sqlite(bib, "biblioteca.db")
con = sqlite3.connect("biblioteca.db")
for titulo, ano in con.execute("SELECT titulo, ano FROM livros WHERE ano < 1950 ORDER BY ano"):
    print(ano, titulo)
print(con.execute("SELECT COUNT(*) FROM livros").fetchone()[0])
con.close()

1888 Os Maias
1901 A Cidade e as Serras
5


# Desafio 5: relatório num ficheiro de texto

In [8]:
totais = {}
for livro in bib.todos():
    totais[livro.autor] = totais.get(livro.autor, 0) + livro.exemplares

mais_antigo = sorted(bib.todos(), key=lambda l: l.ano)[0]

with open("relatorio.txt", "w", encoding="utf-8") as f:
    for autor in sorted(totais):
        f.write(f"{autor}: {totais[autor]} exemplares\n")
    f.write(f"Mais antigo: {mais_antigo.titulo} ({mais_antigo.ano})\n")

with open("relatorio.txt", encoding="utf-8") as f:
    print(f.read(), end="")

Eça de Queirós: 4 exemplares
José Saramago: 6 exemplares
Mais antigo: Os Maias (1888)
